# Proyecto: Análisis de Estacionalidad en Ventas de Automóviles por Marca
**Materia:** Prácticas y Herramientas de Big Data  
**Integrantes:**
* Jose Alexander Quishpe Reinoso
* Julián Garofalo

## Explicación paso a paso
Este notebook está estructurado para llevar a cabo el análisis exploratorio y estadístico del dataset *Car Sales Data*, el cual contiene aproximadamente 2,500,000 de registros. El propósito fundamental es identificar si las ventas y los ingresos de las diferentes marcas de vehículos presentan un comportamiento estacional o si se distribuyen de manera uniforme a lo largo del año fiscal.

El flujo metodológico desarrollado paso a paso comprende:
1. **Extracción y simulación de ingesta:** Configuración del entorno de Big Data en Colab y simulación de consumo de API.
2. **Limpieza y preprocesamiento de millones de filas:** Remoción de duplicados, análisis de nulos y transformaciones de tipos de datos.
3. **Enriquecimiento temporal (Feature Engineering):** Extracción de componentes clave como meses, semanas ISO y trimestres a partir de la fecha.
4. **Formulación de Hipótesis:** Planteamiento formal de los supuestos estacionales de negocio.
5. **Preparación de Estructuras para Modelos:** Organización del espacio para la aplicación de modelos estadísticos y de descomposición de series de tiempo.

---
## Importación de librerías

Antes de comenzar, importamos todas las librerías necesarias para el proyecto. Cada librería cumple un rol específico:

| Librería | Para qué se usa |
|----------|----------------|
| `kagglehub` | Conectarse a la API oficial de Kaggle y cargar el dataset |
| `pandas` | Manipular y limpiar los datos en tablas |
| `numpy` | Operaciones matemáticas y manejo de valores nulos |
| `matplotlib` | Crear gráficos estáticos |
| `seaborn` | Gráficos estadísticos más visuales |
| `sklearn` | Modelos de machine learning |
| `warnings` | Suprimir advertencias innecesarias en la salida |

In [1]:
# ── Acceso al dataset de Kaggle ────────────────────────────────────
import kagglehub                                        # Librería oficial de Kaggle para cargar datasets
from kagglehub import KaggleDatasetAdapter              # Adaptador para cargar directo como DataFrame

# ── Manipulación de datos ───────────────────────────────────────────
import pandas as pd                      # Tablas de datos (DataFrames)
import numpy as np                       # Operaciones numéricas y manejo de NaN

# ── Visualización ──────────────────────────────────────────────────
import matplotlib.pyplot as plt          # Gráficos básicos (barras, líneas, histogramas)
import seaborn as sns                    # Gráficos estadísticos sobre matplotlib

# ── Preprocesamiento (Scikit-learn) ────────────────────────────────
from sklearn.preprocessing import LabelEncoder        # Convierte texto a números para los modelos
from sklearn.preprocessing import StandardScaler      # Escala variables numéricas a la misma magnitud
from sklearn.model_selection import train_test_split  # Divide datos en entrenamiento y prueba

# ── Modelos de Machine Learning ────────────────────────────────────
from sklearn.linear_model import LinearRegression     # Modelo 1: Regresión Lineal
from sklearn.ensemble import RandomForestRegressor    # Modelo 2: Random Forest
from sklearn.ensemble import GradientBoostingRegressor # Modelo 3: Gradient Boosting

# ── Métricas de evaluación ─────────────────────────────────────────
from sklearn.metrics import mean_absolute_error       # Error absoluto medio
from sklearn.metrics import mean_squared_error        # Error cuadrático medio
from sklearn.metrics import r2_score                  # Coeficiente de determinación R²

# ── Configuración general ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')        # Oculta advertencias para mantener la salida limpia

# Configuración visual de los gráficos
plt.style.use('seaborn-v0_8-whitegrid') # Estilo limpio con cuadrícula para los gráficos
pd.set_option('display.max_columns', None) # Muestra todas las columnas al imprimir el DataFrame

print('✅ Todas las librerías importadas correctamente')

✅ Todas las librerías importadas correctamente


---
## 3. Extracción de Datos

### Explicación técnica
En esta sección se consume la **API oficial de Kaggle** mediante la librería `kagglehub`, que permite cargar el dataset directamente en un DataFrame de pandas sin necesidad de descargarlo ni subirlo manualmente. 

Para cumplir con las mejores prácticas y estándares de seguridad de proyectos de Big Data, la extracción utiliza un adaptador de lectura directa en memoria y se autentica de forma segura mediante variables de entorno configuradas previamente en el sistema (Secrets de GitHub).

### Detalles de conexión
* **Librería utilizada:** `kagglehub`.
* **Dataset ID:** `suraj520/car-sales-data`.
* **Archivo:** `car_sales_data.csv`.
* **Adaptador de datos:** `KaggleDatasetAdapter.PANDAS` (convierte el flujo directamente a DataFrame)
* **Credenciales:** Variables de entorno `KAGGLE_USERNAME` y `KAGGLE_KEY`.

### Variables obtenidas del dataset
Al realizar la extracción, se obtienen las siguientes columnas estructuradas:

* `Date`: Fecha exacta de la transacción (clave para el análisis de series temporales).
* `Salesperson`: Nombre del asesor comercial.
* `Customer Name`: Nombre del comprador.
* `Car Make`: Fabricante del vehículo (Toyota, Honda, Ford, Chevrolet, Nissan).
* `Car Model`: Modelo específico del auto.
* `Car Year`: Año de fabricación del vehículo.
* `Sale Price`: Precio de venta final en USD (variable de ingresos).
* `Commission Rate`: Porcentaje de comisión asignado.
* `Commission Earned`: Monto total de comisión generado.

**Cantidad de registros estimados:** 2,500,000 filas.


In [ ]:
# ── PASO 1: Validación de credenciales de entorno ─────────────────

# 1. import os: Esta librería permite a Python interactuar directamente con el 
# sistema operativo de la máquina (en este caso, tu contenedor de Codespace).
# La necesitamos para acceder a la memoria donde GitHub guarda los Secrets.
import os 

# 2. os.environ: Es un diccionario que contiene todas las variables del sistema.
# 3. .get(): Busca la variable. Si no la encuentra, en lugar de dar un error fatal, 
# simplemente devuelve el valor por defecto ('❌ No configurado').
print('KAGGLE_USERNAME:', os.environ.get('KAGGLE_USERNAME', '❌ No configurado'))

# 4. Operador ternario (if-else en una línea): Evalúa si la clave existe.
# Por ciberseguridad, NUNCA imprimimos el valor real de la clave (os.environ.get('KAGGLE_KEY')).
# Solo verificamos su existencia para confirmar que el entorno está listo.
print('KAGGLE_KEY:     ', '✅ Configurado' if os.environ.get('KAGGLE_KEY') else '❌ No configurado')

KAGGLE_USERNAME: josquishpereinoso
KAGGLE_KEY:      ✅ Configurado


In [7]:
# ── PASO 2: Cargar el dataset directo desde Kaggle ─────────────────

# 1. kagglehub.dataset_load(): Es la función principal de la nueva librería. 
# Se encarga de autenticarse automáticamente usando las credenciales validadas
# en el paso anterior y de buscar el dataset directamente en los servidores de Kaggle.
df = kagglehub.dataset_load(
    
    # 2. KaggleDatasetAdapter.PANDAS: Este adaptador es clave en proyectos de Big Data.
    # En lugar de descargar un archivo físico (.csv o .zip) al disco duro,
    # lee el flujo de datos y lo convierte inmediatamente en un DataFrame de pandas en memoria.
    KaggleDatasetAdapter.PANDAS,
    
    # 3. 'suraj520/car-sales-data': Es el identificador único del repositorio en Kaggle.
    # Se compone del nombre de usuario del creador (suraj520) y el nombre del proyecto.
    'suraj520/car-sales-data',
    
    # 4. 'car_sales_data.csv': Especifica qué archivo exacto queremos extraer,
    # ya que un repositorio de Kaggle puede contener múltiples archivos de datos.
    'car_sales_data.csv'
)

# ── PASO 3: Variables obtenidas ────────────────────────────────────
# Imprimimos los nombres de las columnas para verificar la estructura extraída.
# tolist() convierte el objeto Index de pandas en una lista estándar de Python 
# para que se imprima de forma horizontal y legible.
print('Columnas del dataset:')
print(df.columns.tolist())

# ── PASO 4: Cantidad de registros ──────────────────────────────────
# len(df) cuenta las filas, validando que logramos extraer el millón de registros.
# df.shape devuelve una tupla con (filas, columnas), dando una visión de la magnitud estructural.
print(f'\n Registros extraídos: {len(df):,}') # El [:,] añade comas para leer mejor los miles
print(f' Dimensiones (filas x columnas): {df.shape}')

# ── PASO 5: Vista previa del dataset ───────────────────────────────
# head() es una función de pandas que muestra por defecto las primeras 5 filas.
# Funciona como evidencia visual de que los datos no solo se cargaron, 
# sino que tienen el formato correcto (fechas, textos y números alineados).
df.head()

Columnas del dataset:
['Date', 'Salesperson', 'Customer Name', 'Car Make', 'Car Model', 'Car Year', 'Sale Price', 'Commission Rate', 'Commission Earned']

 Registros extraídos: 2,500,000
 Dimensiones (filas x columnas): (2500000, 9)


,Date,Salesperson,Customer Name,Car Make,Car Model,Car Year,Sale Price,Commission Rate,Commission Earned
0,2022-08-01,Monica Moore MD,Mary Butler,Nissan,Altima,2018,15983,0.070495,1126.73
1,2023-03-15,Roberto Rose,Richard Pierce,Nissan,F-150,2016,38474,0.134439,5172.40
2,2023-04-29,Ashley Ramos,Sandra Moore,Ford,Civic,2016,33340,0.114536,3818.63
3,2022-09-04,Patrick Harris,Johnny Scott,Ford,Altima,2013,41937,0.092191,3866.20
4,2022-06-16,Eric Lopez,Vanessa Jones,Honda,Silverado,2022,20256,0.113490,2298.85
